# 04 — 필수 유전자 분류 입력 특성 절제 실험(ablation) (`results/esscls_ablation`)

> **GPU 상주** — ESM 토큰 임베딩까지 모든 입력을 GPU 메모리에 올려 배치를 GPU 안에서 색인으로 만든다. VRAM 180 GB급 GPU용 (ESM 토큰 bf16 약 145 GiB).

`03-train_esscls.ipynb` 와 같은 데이터 · 폴드 분할 · 모델 · 학습 · 평가 규약에서 **입력 조합만 바꾼** 13가지 ablation을 차례로
genus 단위 중첩 교차검증으로 평가한 뒤 ablation별 최종 모델을 훈련하고, 결과를 종합 csv로 모은다.

| ablation | 입력 | 연결 벡터 폭 | 전체 입력 대비 |
|---|---|---|---|
| `full` (기준) | seq + esm + prot + bio + ctx × 3 | 2712 | — (`results/esscls_50`) |
| `noctx` | seq + esm + prot + bio | 2508 | 문맥 특성 3종 제외 |
| `nobio` | seq + esm + prot + ctx × 3 | 2572 | Biopython 특성 제외 |
| `noesm` | seq + prot + bio + ctx × 3 | 1560 | ESM 토큰 제외 |
| `noseq` | esm + prot + bio + ctx × 3 | 2648 | 정수 서열 제외 |
| `noder` | seq + esm + prot | 2368 | 파생 변수(bio · ctx × 3) 제외 |
| `noemb` | seq + bio + ctx × 3 | 408 | ESM 토큰 · ProteomeLM 제외 |
| `seq-prot` | seq + prot | 1216 | 정수 서열 + ProteomeLM만 |
| `esm-prot` | esm + prot | 2304 | 두 임베딩만 (정수 서열 · 파생 변수 제외) |
| `prot` | prot | 1152 | ProteomeLM만 |
| `poolesm` | seq + esm_mean + prot + bio + ctx × 3 | 2712 | ESM 토큰 → 토큰 평균 벡터 |
| `poolesm-noseq` | esm_mean + prot + bio + ctx × 3 | 2648 | ESM 토큰 → 토큰 평균 벡터, 정수 서열 제외 |
| `poolesm-noder` | seq + esm_mean + prot | 2368 | ESM 토큰 → 토큰 평균 벡터, 파생 변수 제외 |
| `poolesm-prot` | esm_mean + prot | 2304 | ESM 토큰 → 토큰 평균 벡터, 두 임베딩만 |

- esm = ESM 토큰 임베딩, seq = 정수 인코딩 서열, esm_mean = ESM 토큰 평균 임베딩, ctx × 3 = `ctx` · `ctxp` · `ctxb`
- 46균주 · 28 genus · 195,354 샘플 (필수 18,037 · 비필수 177,317)

진행: 1 데이터 → 2 폴드 분할 → 3 모델 · 손실 · 지표 → 4 ablation별 중첩 교차검증 · 최종 모델 · 확률 보정 → 5 ablation 종합

In [1]:
# !pip install torch scikit-learn h5py pandas matplotlib tqdm biopython
# !pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # VRAM 조각화 완화 (torch 불러오기 전에 설정)
import time, numpy as np, pandas as pd, torch
from proteomefit import batching, data, features, folds, cv
from proteomefit.config import Config, setup_torch
from proteomefit.models import ds_blocks, param_summary
from proteomefit.tasks import EssTask

cfg = Config(
    out_dir    = '../results/esscls_ablation',
    full_dir   = '../results/esscls_50',   # 전체 입력 모델 결과 (비교 기준, 03 노트북)
    droprate   = 0.5,                        # 연결 벡터 · MLP 은닉층 · PMA 드롭아웃
    mlp_hidden = (640, 320),                 # MLP 은닉 유닛 수 (길이 = 은닉층 수)
    lr         = 1e-3,                       # 학습률 (AdamW)
    min_delta  = 0.01,                       # 검증 MCC 개선 인정 하한
)                                            # 나머지 설정 = proteomefit/config.py 기본값

# ablation 이름 -> 입력 (esm = ESM 토큰, seq = 정수 서열, 나머지 = 데이터셋 블록). 적힌 순서대로 연결하며 seq · esm 은 데이터셋 블록보다 앞에 둔다
ABLATIONS = {
    'noctx':         ('seq', 'esm', 'prot', 'bio'),
    'nobio':         ('seq', 'esm', 'prot', 'ctx', 'ctxp', 'ctxb'),
    'noesm':         ('seq', 'prot', 'bio', 'ctx', 'ctxp', 'ctxb'),
    'noseq':         ('esm', 'prot', 'bio', 'ctx', 'ctxp', 'ctxb'),
    'noder':         ('seq', 'esm', 'prot'),
    'noemb':         ('seq', 'bio', 'ctx', 'ctxp', 'ctxb'),
    'seq-prot':      ('seq', 'prot'),
    'esm-prot':      ('esm', 'prot'),
    'prot':          ('prot',),
    'poolesm':       ('seq', 'esm_mean', 'prot', 'bio', 'ctx', 'ctxp', 'ctxb'),
    'poolesm-noseq': ('esm_mean', 'prot', 'bio', 'ctx', 'ctxp', 'ctxb'),
    'poolesm-noder': ('seq', 'esm_mean', 'prot'),
    'poolesm-prot':  ('esm_mean', 'prot'),
}
FULL_INPUTS = ('seq', 'esm', 'prot', 'bio', 'ctx', 'ctxp', 'ctxb')   # 비교 기준(전체 입력) 모델의 입력
NEED = set().union(*ABLATIONS.values())      # 불러올 입력 종류
setup_torch()

print('device:', cfg.device, f'| {cfg.inner_k} inner folds (val genus matched to test extrapolation distance) '
      '| loss weighted BCE | early stop on val MCC')
print(f'classes: {EssTask.classes} | evaluated on all samples | threshold {cfg.thresh}')
print('ablations:', ', '.join(ABLATIONS))
print('autocast:', cfg.amp_dtype, '| compile:', cfg.compile, '| all inputs GPU-resident')
if cfg.device != 'cpu':
    p = torch.cuda.get_device_properties(0)
    print(f'gpu: {p.name} | VRAM {p.total_memory / 2**30:.0f} GiB | torch {torch.__version__}')

device: cuda:0 | 3 inner folds (val genus matched to test extrapolation distance) | loss weighted BCE | early stop on val MCC
classes: ('non-essential', 'essential') | evaluated on all samples | threshold 0.5
ablations: noctx, nobio, noesm, noseq, noder, noemb, seq-prot, esm-prot, prot, poolesm, poolesm-noseq, poolesm-noder, poolesm-prot
autocast: torch.bfloat16 | compile: True | all inputs GPU-resident
gpu: NVIDIA B200 | VRAM 178 GiB | torch 2.8.0+cu128


## 1. 데이터

모델 입력은 가변 길이 입력 2종과 고정 길이 **데이터셋** 1종이다. ablation마다 이 중 일부만 쓴다.

### 1.1 가변 길이 입력 — 각각 PMA로 벡터 하나로 축약

| 이름 | 형태 | 출처 | 내용 |
|---|---|---|---|
| ESM 토큰 임베딩 | `[T, 1152]` | `emb_esm_600m.h5` | ESM-C 600M 토큰별 표현 (BOS/EOS 포함) |
| 정수 인코딩 서열 | `[L]` | `seq_aa` | 0 = 패딩, 1 ~ 20 = 아미노산, 21 = 그 외. 앞 `MAX_LEN` 잔기 |

### 1.2 데이터셋 블록

| 블록 | 열 | 내용 |
|---|---|---|
| `esm_mean` | 1152 | ESM 토큰 평균 임베딩, 균주 단위 z-정규화 (`poolesm` 계열에서 ESM 토큰 대신 사용) |
| `prot` | 1152 | ProteomeLM 임베딩, 균주 단위 z-정규화 |
| `bio` | 140 | Biopython 서열 특성 (물리화학 12 · 조성 20 · N말단 20 · C말단 20 · 코돈 빈도 64 · GC 4), 균주 단위 z-정규화 |
| `ctx` | 68 | 문맥 특성, ESM 평균 임베딩 유사도 공간 |
| `ctxp` | 68 | 문맥 특성, ProteomeLM 임베딩 유사도 공간 |
| `ctxb` | 68 | 문맥 특성, 두 공간을 이은 유사도 공간 |

**문맥 특성** — 질의 샘플마다 46개 균주 각각에서 가장 유사한 단백질 1개를 찾고, 그중 **참조 균주** 의 것만
유사도 상위 $k \in \{1, 3, 5, 10, 20\}$ 개로 요약한다. 참조 균주는 **시험 genus · 검증 genus · 자기 genus
어디에도 속하지 않는 균주** 이므로, 데이터셋은 내부 폴드마다 다시 만든다.

| 묶음 | 열 | 내용 |
|---|---|---|
| `cons_*` | 9 | 보존 정도 — 상위 $k$ 평균 코사인, 1위와 3위 · 10위의 격차, 코사인 표준편차, 참조 수 (라벨 미사용) |
| `otf_*` | 25 | 참조 단백질의 적합도 · 필수 여부 요약 (원본 척도) |
| `otr_*` | 12 | 같은 요약, 균주 내 순위 척도 |
| `rbh_*` | 14 | 상호 최적 히트만으로 한 요약 |
| `para_*` | 8 | 자기 게놈 안 유사 단백질의 코사인 요약 (라벨 미사용) |

In [3]:
# 라벨·메타 - fit_mean 이 공란(미상)인 샘플만 제외 (n_all 217,468 -> n 195,354)
D    = data.load_samples(cfg.base_dir, cfg.label_col, finite_only=False)
task = EssTask(D, cfg)                                          # 라벨 y [n] (1 = 필수) · GPU 라벨
y    = task.y

n_pos_org = pd.Series(y).groupby(D.org).sum()
NO_ESS    = n_pos_org.index[n_pos_org == 0].to_numpy()          # 필수 샘플이 없는 균주
print(f'samples:{len(D.lab):,} / {len(D.base):,} (dropped unknown {len(D.base) - len(D.keep):,})')
print(f'  essential {int(y.sum()):,} ({y.mean() * 100:.2f}%) | non-essential {int((y == 0).sum()):,}')
print(f'no essential samples: {len(NO_ESS)} organisms ({", ".join(map(str, NO_ESS))})')

samples:195,354 / 217,468 (dropped unknown 22,114)
  essential 18,037 (9.23%) | non-essential 177,317
no essential samples: 4 organisms (Burk376, Magneto, Putida, RalstoniaUW163)


In [4]:
# Biopython 특성 - 전 유전자에서 계산 -> 균주별 z-정규화 -> 사용 샘플만 [n, 140]
seq_aa_all = D.base.seq_aa.fillna('').to_numpy()
BIOF  = features.bio_features(seq_aa_all, D.base.seq_dna.fillna('').to_numpy())   # [n_all, 140]
D.BIO = features.zscore_per_org(BIOF, D.org_code_all)[D.keep]
del BIOF

# 정수 인코딩 서열 - 앞 max_len 잔기를 한 배열에 이어 붙인다 (샘플 i = seq_flat[seq_off[i]:seq_off[i+1]], 훈련 때 배열째 GPU 상주)
D.seq_flat, D.seq_off, D.seq_len, raw_len = features.encode_seqs(seq_aa_all[D.keep], cfg.max_len)
print(f'seq residues {len(D.seq_flat):,} (max {int(D.seq_len.max())}, truncated {int((raw_len > cfg.max_len).sum())})')
print('bio', D.BIO.shape)

biopython: 100%|██████████| 217468/217468 [00:37<00:00, 5812.40it/s]


seq residues 66,920,905 (max 4096, truncated 80)
bio (195354, 140)


### 직교상동체 목록

전체 217,468개 단백질(라벨 미상 포함) 각각에 대해 46개 균주마다 코사인이 가장 높은 단백질의 번호와 그 값을
구한다(`ORT`). 임베딩은 차원별 표준화 후 L2 정규화(백색화)해 내적 = 코사인으로 만든다. 균주 프로테옴 표현
`ORG_VEC`(2절)도 여기서 구한다.

In [5]:
prot_raw, esm_mean_raw = data.load_embeddings(cfg.prot_h5, cfg.mean_h5, D)        # [n_all, 1152] x 2
D.PROT    = features.zscore_per_org(prot_raw, D.org_code_all)[D.keep]             # 데이터셋 prot 블록 [n, 1152]
D.EMEAN   = features.zscore_per_org(esm_mean_raw, D.org_code_all)[D.keep]         # 데이터셋 esm_mean 블록 [n, 1152]
D.ORG_VEC = pd.DataFrame(prot_raw).groupby(D.org_code_all).mean().to_numpy()      # 균주 프로테옴 표현 [n_org, 1152]

# 직교상동체 목록 3종 (ESM · ProteomeLM · 둘을 이은 공간) - 값 = (sim, idx, para)
D.ORT = features.ortholog_lists(esm_mean_raw, prot_raw, D.org_code_all, D.n_org, max(cfg.topk), cfg.sim_chunk, cfg.device)
del prot_raw, esm_mean_raw
print('ortholog lists:', {k: (v[0].shape, v[2].shape) for k, v in D.ORT.items()})

para both: 100%|██████████| 46/46 [00:00<00:00, 385.77it/s]


ortholog lists: {'esm': ((217468, 46), (217468, 20)), 'prot': ((217468, 46), (217468, 20)), 'both': ((217468, 46), (217468, 20))}


In [6]:
# ESM 토큰별 임베딩 - 'esm' 을 쓰는 ablation 이 있을 때만 GPU 한 배열에 이어 붙인다 (샘플 i = esm[esm_off[i]:esm_off[i+1]], bf16)
D.esm = D.esm_off = D.esm_len = None
if 'esm' in NEED:
    D.esm, D.esm_off, D.esm_len = data.load_esm_tokens(cfg.esm_h5, D, cfg.device, cfg.amp_dtype, cfg.esm_chunk, cfg.gpu_reserve)
    print(f'ESM tokens:{len(D.esm):,} x {D.esm.shape[1]} = {D.esm.nbytes / 2**30:.1f} GiB GPU (bf16) | '
          f'free after load {torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB')

ESM -> GPU: 100%|██████████| 107/107 [03:24<00:00,  1.91s/it]

ESM tokens:67,311,613 x 1152 = 144.4 GiB GPU (bf16) | free after load 33.1 GiB


In [7]:
D.block_dim = features.block_dims(D, cfg.seq_dim)                # 입력별 연결 벡터 폭
W_ABL = max(map(len, ABLATIONS))
for name, inp in {'full': FULL_INPUTS, **ABLATIONS}.items():
    print(f'{name:<{W_ABL}}  width {sum(D.block_dim[b] for b in inp):>5}  = ' + ' + '.join(f'{b}({D.block_dim[b]})' for b in inp))
print(f'organisms:{D.lab.org_id.nunique()} | genus:{D.lab.genus.nunique()}')
sh = pd.Series(y).groupby(D.gen).mean()                          # genus별 필수 비율 [n_fold]
print(f'per-genus essential share (min~max): {sh.min():.3f}~{sh.max():.3f}')

full           width  2712  = seq(64) + esm(1152) + prot(1152) + bio(140) + ctx(68) + ctxp(68) + ctxb(68)
noctx          width  2508  = seq(64) + esm(1152) + prot(1152) + bio(140)
nobio          width  2572  = seq(64) + esm(1152) + prot(1152) + ctx(68) + ctxp(68) + ctxb(68)
noesm          width  1560  = seq(64) + prot(1152) + bio(140) + ctx(68) + ctxp(68) + ctxb(68)
noseq          width  2648  = esm(1152) + prot(1152) + bio(140) + ctx(68) + ctxp(68) + ctxb(68)
noder          width  2368  = seq(64) + esm(1152) + prot(1152)
noemb          width   408  = seq(64) + bio(140) + ctx(68) + ctxp(68) + ctxb(68)
seq-prot       width  1216  = seq(64) + prot(1152)
esm-prot       width  2304  = esm(1152) + prot(1152)
prot           width  1152  = prot(1152)
poolesm        width  2712  = seq(64) + esm_mean(1152) + prot(1152) + bio(140) + ctx(68) + ctxp(68) + ctxb(68)
poolesm-noseq  width  2648  = esm_mean(1152) + prot(1152) + bio(140) + ctx(68) + ctxp(68) + ctxb(68)
poolesm-noder  width  2368  = seq(

## 2. 폴드 분할 — 외부 · 내부 모두 genus 단위

$$\text{test}_{k} = G_k, \qquad \text{val}_{k,j} = G_{v_{k,j}}, \qquad \text{train}_{k,j} = \{\, i \mid \mathrm{gid}(i) \notin \{k,\ v_{k,j}\} \,\}$$

- $G_g$ — genus $g$ 의 전 샘플
- **외부 폴드** $k$ — 테스트 genus $k$ (28개, genus 단위 리브-원-아웃 교차검증(Leave-One-Out Cross-Validation, LOOCV))
- **내부 폴드** $j$ — 검증 genus $v_{k,j}$ 하나 (`INNER_K` = 3개). 검증 genus는 훈련에서 통째로 빠진다
- 내부 폴드는 **성능 평가 전용** 이다. 내부 폴드 모델 3개를 각각 테스트 genus에 평가한 지표의 평균이 그 genus의 성능이다. 모든 ablation이 같은 분할을 쓴다

### 검증 genus 선택 — 시험 genus와 외삽 거리가 비슷한 genus

$$\mathbf{u}_o = \frac{1}{n_o} \sum_{i \in o} \mathbf{e}_i, \qquad \mathbf{v}_g = \frac{1}{\lvert O_g \rvert} \sum_{o \in O_g} \mathbf{u}_o, \qquad s(k, g) = \cos\big(\mathbf{v}_k - \bar{\mathbf{v}},\ \mathbf{v}_g - \bar{\mathbf{v}}\big)$$

- $\mathbf{e}_i$ — 유전자 $i$ 의 ProteomeLM 임베딩 (z-정규화 전 원본)
- $\mathbf{u}_o$ — 균주 $o$ 의 프로테옴 표현 = 그 균주 전 유전자 $n_o$ 개의 평균 (`ORG_VEC`)
- $\mathbf{v}_g$ — genus $g$ 의 표현 = 그 genus 균주 집합 $O_g$ 의 $\mathbf{u}_o$ 평균 (`GVEC`)
- $\bar{\mathbf{v}}$ — 전 genus 표현의 평균 (중심화)
- $s(k, g)$ — 시험 genus $k$ 와 genus $g$ 의 코사인 유사도 (`GSIM`)

$$s^{\mathrm{nn}}(g \mid X) = \max_{h \notin X \cup \{g\}} s(g, h), \qquad d(v) = \big\lvert s^{\mathrm{nn}}(v \mid \{k\}) - s^{\mathrm{nn}}(k \mid \varnothing) \big\rvert$$

- $s^{\mathrm{nn}}(g \mid X)$ — genus $g$ 와, $X$ 를 뺀 나머지 genus 중 가장 닮은 것의 유사도 = **외삽 거리** (`nn_sim`)
- $s^{\mathrm{nn}}(k \mid \varnothing)$ — 시험 genus 의 외삽 거리, $s^{\mathrm{nn}}(v \mid \{k\})$ — 내부 폴드에서 검증 genus $v$ 의 외삽 거리 ($k$ 제외)
- 후보를 $d(v)$ 오름차순으로 세워 앞의 `INNER_K` 개를 검증 genus 로 쓴다
- 후보 = 시험 genus와 그 최근접 genus를 뺀 genus 중 검증 샘플에 양·음성이 모두 있는 균주가 하나 이상인 것

In [8]:
# genus 표현 -> 중심화 코사인 유사도 -> 검증 genus 선택 (후보 = 검증 샘플에 양·음성이 모두 있는 균주가 하나 이상인 genus)
S = folds.make_splits(D.gen, D.org2genus, D.ORG_VEC, cfg.inner_k, candidate=task.val_candidate)
folds.print_splits(S, D.lab, cfg.inner_k, y=y)                   # 분할 요약 + 누출 점검 (warn 이 없으면 정상)

nested CV: 28 outer x 3 inner (genus-level) = 84 trainings / 195,354 samples
val-genus candidates 26/28 (excluded: Paraburkholderia, Magnetospirillum)

fold01 Pseudomonas     : test 48,716(10org, pos  7.5%) | val Acidovorax(+0.17, nn 0.41, 4,428) Desulfovibrio(-0.32, nn 0.42, 5,947) Marinobacter(+0.13, nn 0.42, 4,203)
fold02 Ralstonia       : test 16,451(4org, pos  8.4%) | val Azospirillum(+0.11, nn 0.75, 4,518) Dickeya(-0.10, nn 0.76, 11,870) Shewanella(-0.50, nn 0.61, 14,788)
fold03 Shewanella      : test 14,788(4org, pos  9.4%) | val Sphingomonas(-0.23, nn 0.60, 3,803) Caulobacter(-0.04, nn 0.60, 3,745) Dyella(-0.41, nn 0.58, 3,982)
fold04 Dickeya         : test 11,870(3org, pos 10.3%) | val Azospirillum(-0.27, nn 0.75, 4,518) Ralstonia(-0.10, nn 0.69, 16,451) Cupriavidus(-0.18, nn 0.69, 6,847)
fold05 Cupriavidus     : test  6,847(1org, pos  6.9%) | val Azospirillum(-0.01, nn 0.75, 4,518) Dickeya(-0.18, nn 0.76, 11,870) Shewanella(-0.39, nn 0.61, 14,788)
fold06 Desulfovibrio   : tes

## 3. 모델 · 손실 · 지표

### 구조

```
정수 서열 [B,L] ─── 임베딩 + 위치 ─ PMA ── [B,64] ───┐   (입력에 'seq' 가 있을 때)
ESM 토큰 [B,T,1152] ── PMA ────────────── [B,1152] ─┼─ concat ─ MLP ─ 로짓 [B]   (입력에 'esm' 이 있을 때)
데이터셋 [B,d] ──────────────────────────────────────┘   (입력의 데이터셋 블록만)
```

- 다중 헤드 어텐션 풀링(Pooling by Multihead Attention, PMA) — 학습 가능한 시드 벡터 1개를 쿼리로, 토큰열을 어텐션 가중 평균해 벡터 하나로 줄인다. 키 투영은 쿼리 쪽으로 접고 값 투영은 가중 평균 뒤에 적용한다(`nn.MultiheadAttention` 과 동치)
- 다층 퍼셉트론(Multi-Layer Perceptron, MLP) — `Dropout → [Linear → GELU → Dropout] × len(MLP_HIDDEN) → Linear(1)`. `MLP_HIDDEN = [640, 320]` (연결 벡터 폭 → 640 → 320 → 1). 출력이 로짓 $z$ 이고 $\sigma(z)$ 가 필수일 확률이다
- ablation 입력에 없는 분기는 만들지 않는다. 가변 길이 입력이 없으면 배치는 무작위 순서로 `BATCH` 개씩 묶인다

### 손실 — 가중 이진 교차 엔트로피(Binary Cross-Entropy, BCE)

$$\mathcal{L} = \frac{1}{\sum_i w_i} \sum_i w_i \Big[ -\,p_{+}\, y_i \log \sigma(z_i) - (1 - y_i) \log\big(1 - \sigma(z_i)\big) \Big], \qquad w_i = \frac{1}{n_{\mathrm{org}}(g_i)\, n_{\mathrm{smp}}(o_i)}$$

- $y_i$ — 라벨 (1 = 필수), $z_i$ — 로짓
- $p_{+}$ — 양성 가중 = 훈련 샘플의 음성 수 / 양성 수
- $w_i$ — 분류군 역크기 가중치 (genus마다 같은 비중). $n_{\mathrm{org}}(g_i)$ = 샘플 $i$ 의 genus 안 균주 수, $n_{\mathrm{smp}}(o_i)$ = 그 균주의 샘플 수. 훈련 샘플 평균 1로 정규화한다

### 평가 샘플

테스트 genus의 **전 샘플** 로 평가한다. 검증셋은 검증 genus의 전 샘플이다.

### 평가 지표 — 매튜스 상관계수(Matthews Correlation Coefficient, MCC)

$$MCC = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

MCC는 균주별로 구해 평균한다. 양·음성이 모두 있는 균주만 넣는다(`org_masks` · `in_total`). 조기 종료와 결과 정렬의
기준은 MCC이고, 결과 파일에는 혼동행렬 4칸 · F1 · 재현율 · 정밀도 · 정확도 · ROC AUC · PR AUC 도 함께 남긴다.

### 구현 위치

| 내용 | 모듈 |
|---|---|
| 문맥 특성 · 데이터셋 (`ctx_block` · `build_dataset`) | `proteomefit/features.py` |
| 모델 (`PMA` · `SeqBranch` · `EssNet`) | `proteomefit/models.py` |
| 배치 구성 · GPU 상주 배치 공급 · 추론 | `proteomefit/batching.py` |
| 라벨 · 손실 · 샘플 가중치 · 조기 종료 지표 · 평가 산출물 (`EssTask`) | `proteomefit/tasks.py` |
| 지표 · 그림 | `proteomefit/metrics.py` · `proteomefit/plots.py` |
| 훈련 루프 (`train_model`) | `proteomefit/train.py` |
| 중첩 교차검증 · 최종 모델 · 요약 (`run_cv` · `train_final` · `print_summary`) | `proteomefit/cv.py` |

## 4. ablation별 중첩 교차검증 · 최종 모델 · 확률 보정

ablation마다 `03-train_esscls.ipynb` 4 · 5 · 6절과 같은 과정을 거친다.

1. **중첩 교차검증** — 내부 폴드 $(k, j)$ 마다 그 ablation의 데이터셋 블록으로 데이터셋을 만들고(시험 · 검증 genus를 참조에서 제외,
   훈련 샘플 기준 표준화), 훈련 → best epoch 가중치로 테스트 genus의 전 샘플 평가
2. **최종 모델** — 전체 샘플로 $\lfloor \tilde{\mu} + 0.5 \rfloor$ epoch 훈련. $\tilde{\mu}$ 는 genus별 best epoch(내부 폴드 best epoch 의 중앙값)의 중앙값.
   문맥 특성은 자기 genus만 참조에서 빼고, 표준화 기준은 전체 샘플이다
3. **확률 보정** — 교차검증 예측 확률 → `conf_ess` 확률표 (같은 genus 분할의 중첩 교차검증 + 최종 보정 모델, `proteomefit/calib.py`). 판정 = `conf_ess` ≥ 0.5

| 경로 (`OUT_DIR/`) | 산출물 |
|---|---|
| `<ablation>/fold##_<genus>/inner##/` | `model.pt` · `history.csv` · `history.png` · `metrics_organ.csv` · `pred.npz` (평가 샘플 필수 확률 `prob`, 테스트 genus 샘플 순) |
| `<ablation>/fold##_<genus>/` | `metrics_folds.csv` — 내부 폴드별 1줄 + mean · median · std 줄 |
| `<ablation>/` | `metrics_total.csv` — genus별 1줄 (내부 폴드 지표의 평균. best epoch 은 중앙값 `best_epoch_med` (채택) · 평균 `best_epoch_mean` 을 함께 남긴다, `_sd` = 내부 폴드 간 표준편차) + genus 간 mean · median · std 줄 |
| `<ablation>/` | `model_final.pt` · `history_final.csv` · `history_final.png` — 최종 모델 |
| `<ablation>/fold##_<genus>/inner##/` | `model_calib.pkl` · `pred_calib.npz` · `metrics_organ_calib.csv` · `calibration.png` · `confusion.png` (보정 전 · 후) |
| `<ablation>/` | `metrics_folds_calib.csv` (폴드별) · `metrics_total_calib.csv` · `compare_calib.csv` · `check_calib.csv` · `calibration.png` · `model_final_calib.pkl` · `calibration.json` — 보정 결과 (열 = 원 모델 표 + `ece` · `brier` · `log_loss`) |

`RESUME=True` 면 산출물이 모두 있는 내부 폴드 · 최종 모델 · 보정 폴드(`pred_calib.npz` · `model_calib.pkl`)는 다시 훈련하지 않는다. `ABLATIONS` 에서 항목을 빼면 그 ablation만 건너뛴다.

In [ ]:
import json
from proteomefit import calib

os.makedirs(cfg.out_dir, exist_ok=True)
RES, CRES, t_start = {}, {}, time.perf_counter()                 # ablation -> genus별 결과 DataFrame (원 모델 · 보정)
for name, inputs in ABLATIONS.items():
    _m = task.build_model(inputs, cfg.seed)
    print(f'\n##### ablation {name} | {" + ".join(inputs)} | width {_m.d_in} | params {param_summary(_m)}')
    del _m
    t_abl, out = time.perf_counter(), os.path.join(cfg.out_dir, name)
    RES[name] = cv.run_cv(task, D, S, cfg, inputs, out, name)     # 중첩 교차검증 -> out/metrics_total.csv
    cv.train_final(task, D, S, cfg, inputs, out, RES[name], name)  # 최종 모델 -> out/model_final.pt

    E = calib.load_cv_preds(out, {'ess': y, 'org': D.org}, D.gen)  # 교차검증 예측 [n_pred, 5] (prob, ess, org, genus, inner)
    CRES[name] = calib.nested_cv('ess', E, out, out)[0]            # conf_ess 보정 -> out/metrics_total_calib.csv
    with open(f'{out}/calibration.json', 'w', encoding='utf-8') as f:
        json.dump({'ess': {**calib.final_model('ess', E, CRES[name], out), 'source': out.split('results/')[1]}}, f)

    r, c = RES[name], CRES[name]
    print(f'##### {name} done ({cv.hms(time.perf_counter() - t_abl)}) | genus mean '
          + ' '.join(f'{k} {r[k].mean():.3f}±{r[k].std():.3f}' for k in task.brief_cols)
          + f' | calibrated mcc {c.mcc.mean():.3f}±{c.mcc.std():.3f}')
print(f'\nall ablations done | total elapsed {cv.hms(time.perf_counter() - t_start)}')

## 5. ablation 종합

`FULL_DIR/metrics_total.csv` · `metrics_total_calib.csv`(전체 입력 모델, `03-train_esscls.ipynb` 결과)가 있으면 비교 기준 `full` 로 함께 싣는다.
모든 ablation과 `full` 은 같은 폴드 분할을 쓴다. 종합표는 원 모델 판정(확률 ≥ 0.5)과 보정 판정(`conf_ess` ≥ 0.5) 두 벌이다.

| 파일 (`OUT_DIR/`) | 내용 |
|---|---|
| `ablation_summary.csv` · `ablation_summary_calib.csv` | ablation별 1줄 — 입력, 연결 벡터 폭, 지표가 계산된 genus 수, best epoch 집계 `best_epoch_med` (genus 중앙값들의 중앙값, `n_epoch_final` 의 근거) · `best_epoch_mean` (genus 평균들의 평균) · 최종 모델 epoch 수(`n_epoch_final`, 보정 = 최종 보정 모델 반복 수), 지표의 genus 평균(`_sd` = genus 간 표준편차), `d_mcc`, `n_genus_worse` |
| `ablation_genus_mcc.csv` · `ablation_genus_mcc_calib.csv` | genus × ablation MCC (각 칸 = 그 genus 내부 폴드 MCC 평균) |

- `d_mcc` — ablation의 genus 평균 MCC − `full` 의 genus 평균 MCC
- `n_genus_worse` — MCC가 `full` 보다 낮은 genus 수 (두 쪽 모두 계산된 genus 중)

In [ ]:
summ = cv.summarize_ablations(task, RES, ABLATIONS, FULL_INPUTS, cfg.full_dir, cfg.out_dir,
                              D.block_dim, S.GENERA, cfg.csv_round)            # 원 모델 판정 -> ablation_summary.csv
summ_calib = cv.summarize_ablations(task, CRES, ABLATIONS, FULL_INPUTS, cfg.full_dir, cfg.out_dir,
                                    D.block_dim, S.GENERA, cfg.csv_round, '_calib')   # 보정 판정 -> ablation_summary_calib.csv